# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Authors: {[author['@id'] for author in getattr(metadata, 'author', [])]}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Coverage: {getattr(metadata, 'spatialCoverage', 'N/A')}, {getattr(metadata, 'temporalCoverage', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets and their fields with their @id values
print("Available Record Sets and their Fields:")
record_sets_info = []
for record_set in dataset.record_sets:
    print(f"- Record Set Name: {record_set.name}\n  @id: {record_set.id_}")
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - Field Name: {field.name}   @id: {field.id_}   Type: {field.data_type}")
    record_sets_info.append(record_set.id_)
if not dataset.record_sets:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. (Fields and columns are referenced by their `@id`).

In [ ]:
# Extract data from each record set
record_sets = [rs.id_ for rs in dataset.record_sets]
# Mapping of {record_set_id: record_set_object}
record_set_objects = {rs.id_: rs for rs in dataset.record_sets}
dataframes = {}

for record_set_id in record_sets:
    try:
        # Load records as list (each record references fields by their `@id`)
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")
        continue
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for Record Set: {record_set_id}")

# Show available columns for the first record set (by @id)
if record_sets:
    first_rs = record_sets[0]
    print(f"\nColumns in record set '{first_rs}':\n", dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: e.g., filtering records by a numeric field, normalizing values, grouping. All fields are referenced by their `@id`.

In [ ]:
# Pick a record set and select a numeric field by @id
import numpy as np

# Auto-select a record set and numeric field for demonstration
selected_record_set_id = None
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    df = dataframes.get(rs.id_)
    if df is not None and not df.empty:
        # Attempt to detect numeric fields
        numeric_candidates = []
        group_candidates = []
        for field in getattr(rs, 'fields', []):
            if field.data_type and field.data_type.lower() in ['integer', 'float', 'number']:
                numeric_candidates.append(field.id_)
            if field.data_type and field.data_type.lower() == 'text':
                group_candidates.append(field.id_)
        # Fallback to any column containing numbers if not explicitly typed
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                if col not in numeric_candidates:
                    numeric_candidates.append(col)
        if numeric_candidates:
            selected_record_set_id = rs.id_
            numeric_field_id = numeric_candidates[0]
            if group_candidates:
                group_field_id = group_candidates[0]
            break

if not (selected_record_set_id and numeric_field_id):
    print("No suitable numeric field found for EDA.")
else:
    df = dataframes[selected_record_set_id]
    print(f"Using Record Set: {selected_record_set_id}")
    print(f"Numeric Field for filtering/normalization: {numeric_field_id}")
    if group_field_id:
        print(f"Grouping by: {group_field_id}")
    
    # Convert column to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = np.nanmean(df[numeric_field_id])  # mean as threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}: {len(filtered_df)} records")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std else 0
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping if applicable
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_field_id:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    
    # If grouping field was found, boxplot by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to:
- Load FAIR² dataset metadata and records directly from a Croissant schema URL
- Review the record sets and fields referenced by their `@id`s
- Extract tables and perform simple exploratory analysis (filtering, normalization, and grouping)
- Visualize numeric data distributions and relationships

**Note**: All entities are referenced by their Croissant `@id` values for robust and reproducible exploration. For further analysis, review additional documentation fields and domain metadata in the full Croissant schema.